# ARC-AGI-3 BFS Submission

Architecture per `docs/kaggle-submission.md`:
- Kaggle eval sandbox runs a private `gateway:8001` HTTP service in the same docker network.
- We use `OperationMode.ONLINE` with `ARC_BASE_URL=http://gateway:8001/`.
- The gateway tracks the scorecard; Kaggle scorer overwrites `/kaggle/working/submission.parquet` with real scores in rerun mode.
- In Edit mode we just write a dummy parquet so the notebook is a valid submission.

Strategy here: pure BFS (no CNN). Uses `scripts/bfs_benchmark.py` from our repo. Per-level budget = `scan-timeout + bfs-timeout`. Expected per-game wallclock < 6 min.

## 1. Install arc-agi SDK from competition wheels (no internet)

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

## 2. Bring in our repo from a Kaggle dataset

**Required**: upload `arc-agi-3-pre` as a public Kaggle dataset and attach it here. Adjust `REPO_DATASET` below to the actual slug after attachment.

Quick check: `!ls /kaggle/input/` shows all attached datasets.

In [ ]:
import os, sys, shutil, glob

REPO_DATASET_CANDIDATES = [
    "/kaggle/input/arc-agi-3-pre",
    "/kaggle/input/arc-agi-3-pre-repo",
    "/kaggle/input/arc-agi-3-pre-bfs",
    "/kaggle/input/arc-agi-3-pre-v1",
    "/kaggle/input/arc-agi-3-pre-code",
]
if not any(os.path.isdir(p) for p in REPO_DATASET_CANDIDATES):
    for cand in glob.glob("/kaggle/input/*"):
        if os.path.isfile(os.path.join(cand, "scripts/bfs_benchmark.py")):
            REPO_DATASET_CANDIDATES.insert(0, cand)
            break

REPO_SRC = next((p for p in REPO_DATASET_CANDIDATES if os.path.isdir(p)), None)
REPO_DIR = "/kaggle/working/arc-agi-3-pre"

print("-- /kaggle/input contents --")
!ls /kaggle/input/ 2>/dev/null
print()

if REPO_SRC is None:
    raise SystemExit(
        f"Could not find our repo at any of {REPO_DATASET_CANDIDATES}. "
        f"Attach the dataset to this notebook and add its path above."
    )
print(f"using repo source: {REPO_SRC}")
if not os.path.isdir(REPO_DIR):
    shutil.copytree(REPO_SRC, REPO_DIR)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"repo at {REPO_DIR}")
!git log --oneline -3 2>/dev/null || echo '(not a git checkout — that is fine)'

# === sentinel: confirm dataset has the timeout-fix commit (060aac3+) ===
with open(f"{REPO_DIR}/scripts/bfs_benchmark.py") as f:
    src = f.read()
assert '--total-budget-min' in src, (
    "OLD dataset version — does NOT have the timeout fix.\n"
    "Re-sync your Kaggle dataset to pull the latest commit, then re-run.\n"
    "Without this fix, the notebook will run out of wall-clock and Kaggle will award 0."
)
assert 'finally:' in src, "OLD dataset — close_scorecard not protected by try/finally"
print("✅ dataset has the timeout fix")

## 3. Run BFS solver against the gateway (rerun-mode only)

Sets `ARC_BASE_URL=http://gateway:8001/`, `OPERATION_MODE=online`, `ARC_API_KEY=test-key-123` so `Arcade()` routes through the local gateway. Our `scripts/bfs_benchmark.py` already reads env vars via the SDK.

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for gateway to come up (per StochasticGoose sample notebook pattern)
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    os.environ['ARC_BASE_URL']   = 'http://gateway:8001/'
    os.environ['ARC_API_KEY']    = 'test-key-123'
    os.environ['OPERATION_MODE'] = 'online'

    # Budget reasoning (post-mortem of v1 timeout):
    #   - Kaggle ran us out of wall-clock without ever calling close_scorecard → 0 LB
    #   - Fix 1: --total-budget-min as a HARD cap that triggers early break
    #   - Fix 2: try/finally close_scorecard always runs
    #   - Per-level: scan 3s + bfs 20s = 23s; max 3 levels = 69s/game
    #   - Network overhead via gateway: assume ~3x local → ~3.5 min/game wallclock
    #   - 100 games × 3.5 min = 350 min; cap at 360 min, leave ~30 min slack for any cap up to 7h
    !cd /kaggle/working/arc-agi-3-pre && python scripts/bfs_benchmark.py \
        --tag kaggle-bfs-v2 \
        --max-levels 3 \
        --scan-timeout 3 \
        --bfs-timeout 20 \
        --total-budget-min 360
else:
    print("Not in competition rerun — skipping BFS run. (Will write dummy submission in next cell.)")

## 4. Edit-mode dummy submission

Kaggle requires every notebook submission to leave a `submission.parquet`. In rerun mode the gateway/scorer overwrites this with real scores. In Edit mode we just write a placeholder.

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print(submission)